# Stage 3 — DR Screening Pipeline: DR Severity Grading (image-only, Grad-CAM)

**Updated architecture note:** per the revised design, Stage 2 (lesion/OD/vessel masks) is
**no longer fed into this model**. This notebook trains an **image-only** DR grading model:

```
Stage 1 (quality gate + enhancement)  --->  THIS MODEL (image in, grade + Grad-CAM out)
Stage 2 (segmentation, run separately) ------------------------------> merged into final report only
```

- **Input:** the *quality-gated* image — `Reject`s are dropped before this stage ever sees
  them, `Usable` images are CLAHE + illumination-normalized (same function used in the Stage 1
  notebook), `Good` images pass through unchanged. **This notebook no longer requires a trained
  Stage 1 checkpoint to run:** EyeQ's CSV already carries the ground-truth `quality` label for
  every image, so Stage 3 reads that column directly to decide drop/enhance/pass-through. This
  means Stage 3 and Stage 1 can now be trained independently, in either order. Section 3 has a
  commented-out block to swap in a *real* trained Stage 1 model later, if you want to validate
  against predicted (not ground-truth) quality labels.
- **Output:** ICDR grade (0–4), calibrated referable-DR flag + confidence, and a Grad-CAM
  heatmap. Stage 2's masks are combined with this notebook's output only at **report assembly
  time** (Section 10) — never as a model input.
- **Fast/subset mode for Colab free tier**, same philosophy as the Stage 1 notebook: train on a
  stratified subset with heavy caching + AMP + parallel image loading, but do the metric that
  matters (sensitivity/specificity for referable DR, quadratic-weighted kappa) on the **full
  official EyeQ test split**.

**Output of this notebook** (saved to Drive):
```
stage3_grading/
├── model.pt                 # trained weights
├── config.json               # architecture, class mapping, preprocessing, calibrated threshold
├── training_history.json     # loss/QWK curves
├── eval_report.json          # full-test-set metrics (sensitivity/specificity/QWK/AUC)
└── gradcam_samples/           # a few example overlays for sanity-checking explainability
```

**Before running:** set `TRAIN_CSV`/`TEST_CSV`/image dirs the same way you did in Stage 1
(same EyeQ dataset — this notebook reads both the `DR_grade` column, its training target, and
the `quality` column, used only for the gate/enhance step).


## 1. Setup

In [ ]:
# Colab GPU check — Runtime > Change runtime type > GPU (T4 is fine for EfficientNet-B3 @ 300px with AMP)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
!pip install -q timm scikit-learn opencv-python-headless tqdm


In [ ]:
import os, json, time, random, math, gc
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as T
import timm

from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                              cohen_kappa_score, roc_auc_score, roc_curve)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---- Forced CPU mode ----
# Per project decision: always run on CPU rather than relying on Colab's often-unavailable
# free-tier GPU. This trades speed for reliability -- CPU quota never runs out mid-run, and
# combined with the disk-backed dataset cache below (Section 6/7) this is what keeps a full
# run from crashing with an out-of-RAM session death partway through.
FORCE_CPU = True
device = torch.device("cpu") if FORCE_CPU else torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = (device.type == "cuda")
print("Device:", device, "(forced)" if FORCE_CPU else "")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Config

Same "one place to edit" philosophy as the Stage 1 notebook. Two things worth calling out vs Stage 1:

- `APPLY_QUALITY_PIPELINE` — drives the gate/enhance step (drop Reject, CLAHE-enhance Usable,
  pass Good through) straight from EyeQ's own ground-truth `quality` column. No trained Stage 1
  checkpoint is required for this notebook to run standalone; see Section 3 for the optional
  swap-in once a real Stage 1 checkpoint exists.
- `USE_ORDINAL_HEAD` — CORAL ordinal head (recommended, per the architecture doc — reduces
  adjacent-grade 0↔1 / 2↔3 confusion) vs. a plain softmax head with class-balanced weighted CE
  as a simpler fallback. Both are implemented below; flip the flag to compare.


In [ ]:
CONFIG = {
    # ---- dataset paths (EDIT THESE — same EyeQ dataset as Stage 1) ----
    "TRAIN_CSV": "/content/drive/MyDrive/DR HACK/eyeq_dataset/Label_EyeQ_train.csv",
    "TEST_CSV": "/content/drive/MyDrive/DR HACK/eyeq_dataset/Label_EyeQ_test.csv",
    "TRAIN_IMG_DIR": "/content/drive/MyDrive/DR HACK/eyeq_dataset/train_images",
    "TEST_IMG_DIR": "/content/drive/MyDrive/DR HACK/eyeq_dataset/test_images",

    # ---- output ----
    "OUTPUT_DIR": "/content/drive/MyDrive/dr_screening_pipeline/stage3_grading",

    # ---- model ----
    "MODEL_ARCH": "efficientnet_b3",   # see markdown below for why B3 over ViT-S/MobileNet here
                                        # (auto-downgraded to efficientnet_b0 on CPU-only runtimes, see Section 2b)
    "IMG_SIZE": 300,                    # native-ish resolution for timm's efficientnet_b3
    "NUM_CLASSES": 5,                   # ICDR 0-4
    "USE_ORDINAL_HEAD": True,           # CORAL ordinal head vs plain softmax head

    # ---- training ----
    "BATCH_SIZE": 32,                   # B3 @300px is heavier than B0 @224px -> smaller batch on a T4
    "EPOCHS": 18,
    "LR": 2e-4,
    "WEIGHT_DECAY": 1e-4,
    "LABEL_SMOOTHING": 0.05,            # only used by the softmax-head path
    "EARLY_STOP_PATIENCE": 5,
    "EARLY_STOP_METRIC": "qwk",         # "qwk" (quadratic weighted kappa) or "acc"
    "NUM_WORKERS": 2,                   # Colab free tier ~2 vCPUs
    "PREFETCH_FACTOR": 4,
    "USE_AMP": True,
    "FREEZE_BACKBONE_EPOCHS": 2,        # warm up the (new) head before fine-tuning the whole net
    "TIME_BUDGET_MINUTES": 90,          # soft budget checked by the timing dry-run in Section 7

    # ---- FAST / SUBSET MODE (Colab free tier) ----
    "USE_SUBSET": True,
    "MAX_TRAIN_SAMPLES": 8000,          # stratified subset of the train split (GPU target;
                                         # auto-reduced on CPU-only runtimes, see Section 2b)
    "MAX_VAL_SAMPLES": 1200,            # small stratified subset of test, for fast per-epoch monitoring
    "MIN_PER_CLASS": 150,               # floor so grade 1/3/4 (rare) aren't starved out of the subset —
                                         # raised from 60 so the minority classes are actually learnable

    # ---- class-imbalance handling ----
    # "Effective Number of Samples" re-weighting (Cui et al., CVPR 2019) instead of naive
    # inverse-frequency — naive inverse-freq blows up on the rarest class (grade 4, often <2%
    # of EyePACS/EyeQ) and destabilizes training; effective-number weighting is smoother.
    "CB_BETA": 0.9999,

    # ---- referable-DR operating point ----
    # Functionalities spec: referable DR = ICDR Level 2+ (moderate NPDR or worse).
    # NOTE: the earlier architecture draft calibrated on P(grade>=1) instead — that's a looser
    # (more sensitive, less specific) definition. REFERABLE_GRADE_CUTOFF is the one knob that
    # controls this; set to 1 if you want to match the architecture draft instead of the spec.
    "REFERABLE_GRADE_CUTOFF": 2,
    "TARGET_SENSITIVITY": 0.90,
    "MIN_SPECIFICITY": 0.85,

    # ---- Stage-1-style quality gate/enhance, applied directly from EyeQ's own labels ----
    # No trained Stage 1 checkpoint is required for this: EyeQ's CSV already carries a
    # `quality` column (Good/Usable/Reject) for every image — the exact labels Stage 1 itself
    # is trained on. Stage 3 reads that column directly instead of running a Stage 1 model,
    # so it never needs to wait on Stage 1 being trained first. Swap this for real Stage 1
    # model inference later (see the commented-out block in Section 3) once you have a
    # checkpoint and want to validate that the trained classifier agrees with the ground-truth
    # labels before relying on it in production.
    "APPLY_QUALITY_PIPELINE": True,     # drop Reject, CLAHE+enhance Usable, pass Good through

    "BASELINE_QWK": 0.80,   # a commonly cited quadratic-weighted-kappa ballpark for single-image
                             # EyePACS/APTOS-style DR grading baselines — sanity-check target, not
                             # a strict paper citation; treat as "are we in a reasonable range".
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

Path(CONFIG["OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["OUTPUT_DIR"], "gradcam_samples").mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, indent=2))


### Auto CPU/GPU fallback

The rest of this notebook must not crash just because Colab handed you a CPU-only runtime
(free tier does this sometimes when GPU quota is exhausted). This cell detects that and
automatically scales settings down to something that still finishes — smaller subset, smaller
batch, no AMP (AMP is CUDA-only), fewer workers. It never *raises*; worst case it trains slower.


In [ ]:
if device.type != "cuda":
    _reason = "FORCE_CPU is set" if FORCE_CPU else "no GPU was detected"
    print(f"Running on CPU because {_reason} — using a CPU-safe configuration.\n"
          "Training will be much slower per-image than on a T4, so this trades a lighter "
          "backbone for a LARGER sample count — more data matters more than model capacity "
          "for a 5-class ordinal problem, and EfficientNet-B0 on CPU is still practical at "
          "a few thousand images. It will simply take longer to finish, by design.")
    CONFIG["USE_AMP"] = False                 # torch.cuda.amp is CUDA-only
    CONFIG["MODEL_ARCH"] = "efficientnet_b0"  # B3 on CPU is impractically slow; B0 trains ~4-6x faster
    CONFIG["IMG_SIZE"] = 224                  # matches B0's native resolution, also cheaper per-image
    CONFIG["BATCH_SIZE"] = 16
    CONFIG["MAX_TRAIN_SAMPLES"] = 2500        # was capped at 600 — the real reason your subset was
                                               # ~375/50/74/50/50: raise this the moment you get a GPU
    CONFIG["MAX_VAL_SAMPLES"] = 500
    CONFIG["NUM_WORKERS"] = 0
    CONFIG["EPOCHS"] = 10
    CONFIG["FREEZE_BACKBONE_EPOCHS"] = 2
    CONFIG["TIME_BUDGET_MINUTES"] = 90
else:
    print(f"GPU detected: {torch.cuda.get_device_name(0)} — using the configured settings as-is.")

PIN_MEMORY = device.type == "cuda"   # pinning memory only helps (and only fully works) with CUDA
print(json.dumps(CONFIG, indent=2))


### Why EfficientNet-B3 (not ViT-Small, not MobileNet) for this stage

- **ViT-Small** needs either a lot more labeled data or heavier augmentation/regularization to
  match a pretrained CNN at this scale — with a ~5k-image Colab-free-tier subset it tends to
  underperform a pretrained EfficientNet and takes longer per step at comparable resolution.
- **MobileNetV3 / EfficientNet-B0** (used for the *3-class, coarse* Stage 1 quality gate) has
  less capacity than DR grading needs: separating grade 1 (mild NPDR, a handful of
  microaneurysms) from grade 0 is a much finer-grained visual task than judging overall image
  quality.
- **EfficientNet-B3** is the architecture-doc's stated choice, pretrained on ImageNet, and is
  the best fit for "small-lesion-sensitive, still trains in a single Colab session" — its
  compound-scaled resolution (300px here) keeps small lesions like microaneurysms visible
  without the memory/step-time cost of B5+.


## 3. Load the Stage 1 checkpoint & build the Stage-1-processed image pipeline

This reproduces exactly what the FastAPI Stage 1 service will do to every image before it
reaches this model: classify quality, hard-drop `Reject`, CLAHE+illumination-normalize
`Usable`, pass `Good` through unchanged. Running it here (instead of assuming it) means the
grading model is trained on the same image distribution it will see in production.


In [ ]:
def clahe_illumination_normalize(img_bgr, clip_limit=2.0, tile_grid_size=(8, 8)):
    """Identical to the Stage 1 notebook's enhancement function — kept in sync deliberately."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_eq = clahe.apply(l)
    img_clahe = cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2BGR)

    b_ch, g_ch, r_ch = cv2.split(img_clahe)
    g_blur = cv2.GaussianBlur(g_ch, (0, 0), sigmaX=30)
    g_norm = cv2.divide(g_ch.astype(np.float32), g_blur.astype(np.float32) + 1e-6)
    g_norm = np.clip(g_norm * 128, 0, 255).astype(np.uint8)
    return cv2.merge([b_ch, g_norm, r_ch])


def apply_quality_pipeline(img_bgr, quality_label):
    """Given a raw BGR image and its ground-truth EyeQ quality label, return the image
    Stage 3 would actually receive. 'Reject' rows are filtered out earlier (Section 6) and
    never reach this function at all — mirrors production, where Stage 1 hard-stops Rejects
    before Stage 3 ever sees them."""
    if quality_label == "Usable":
        return clahe_illumination_normalize(img_bgr)
    return img_bgr  # 'Good' passes through unchanged


if CONFIG["APPLY_QUALITY_PIPELINE"]:
    print("Quality gate/enhance ENABLED — reading ground-truth Good/Usable/Reject labels "
          "straight from EyeQ's own CSV (no Stage 1 checkpoint needed): Reject images are "
          "dropped before loading, Usable images get CLAHE + illumination normalization, "
          "Good images pass through unchanged. This is the same decision logic Stage 1 will "
          "enforce at serving time, just driven by ground truth instead of a trained model "
          "while Stage 1 doesn't have a checkpoint yet.")
else:
    print("APPLY_QUALITY_PIPELINE=False — training on raw images, no quality gating/enhancement.")


# ---- Optional: swap in a REAL trained Stage 1 model instead of ground-truth labels -----------
# Once Stage 1 has a checkpoint and you want Stage 3's training distribution to match what a
# *predicted* (not ground-truth) quality gate will actually pass through in production,
# uncomment this block and pass `stage1_model`/`stage1_cfg` into GradingDataset instead of
# relying on the CSV's `quality` column.
#
# def load_stage1_model(output_dir, device=device):
#     output_dir = Path(output_dir)
#     cfg_path, ckpt_path = output_dir / "config.json", output_dir / "model.pt"
#     if not cfg_path.exists() or not ckpt_path.exists():
#         return None, None
#     with open(cfg_path) as f:
#         cfg = json.load(f)
#     m = timm.create_model(cfg["model_arch"], pretrained=False, num_classes=cfg["num_classes"])
#     ckpt = torch.load(ckpt_path, map_location=device)
#     m.load_state_dict(ckpt["model_state_dict"])
#     m.to(device).eval()
#     return m, cfg
#
# stage1_model, stage1_cfg = load_stage1_model("/content/drive/MyDrive/dr_screening_pipeline/stage1_quality")


## 4. Load labels & inspect columns

Same CSV as Stage 1, two target columns pulled this time: `DR_grade` (0–4, what this notebook
trains on) **and** `quality` (Good/Usable/Reject, EyeQ's own ground-truth label — used to drive
the gate/enhance step in Section 3 without needing a trained Stage 1 checkpoint).


In [ ]:
train_df_raw = pd.read_csv(CONFIG["TRAIN_CSV"])
test_df_raw = pd.read_csv(CONFIG["TEST_CSV"])

print("TRAIN columns:", list(train_df_raw.columns))
print("TRAIN shape:", train_df_raw.shape)
train_df_raw.head()


In [ ]:
COLMAP = {
    "filename_col": "image",       # same column as Stage 1
    "grade_col": "DR_grade",       # EyeQ's ICDR grade column (0-4) — edit if your CSV differs
    "quality_col": "quality",      # EyeQ's 0/1/2 quality code — same column Stage 1 trains on
}

GRADE_LABELS = {0: "No DR", 1: "Mild NPDR", 2: "Moderate NPDR", 3: "Severe NPDR", 4: "Proliferative DR"}
QUALITY_LABELS = {0: "Good", 1: "Usable", 2: "Reject"}   # EyeQ's standard quality code mapping

for col in ("filename_col", "grade_col", "quality_col"):
    assert COLMAP[col] in train_df_raw.columns, \
        f"'{COLMAP[col]}' not found — check columns above and update COLMAP"

def _extract(df_raw):
    out = df_raw[[COLMAP["filename_col"], COLMAP["grade_col"], COLMAP["quality_col"]]].copy()
    out.columns = ["filename", "label", "quality_code"]
    out["quality"] = out["quality_code"].map(QUALITY_LABELS)
    return out.drop(columns=["quality_code"])

train_df = _extract(train_df_raw)
test_df = _extract(test_df_raw)

print("Train grade distribution:")
print(train_df["label"].map(GRADE_LABELS).value_counts())
print("\nTest grade distribution:")
print(test_df["label"].map(GRADE_LABELS).value_counts())

print("\nTrain quality distribution (drives the gate/enhance step, Section 3):")
print(train_df["quality"].value_counts())


## 5. Stratified subsampling for fast Colab training

Same pattern as Stage 1: keep `train_df_full` / `test_df_full` around for the baseline-
comparable final evaluation, train on a smaller stratified `train_df`, with `MIN_PER_CLASS`
protecting the rare `Proliferative DR` (grade 4) class from being sampled down to near-zero.


In [ ]:
train_df_full = train_df.copy()
test_df_full = test_df.copy()

def stratified_subsample(df, max_samples, min_per_class, seed):
    if not CONFIG["USE_SUBSET"] or len(df) <= max_samples:
        return df.reset_index(drop=True)

    class_counts = df["label"].value_counts()
    proportional = (class_counts / class_counts.sum() * max_samples).round().astype(int)
    target = {c: max(min_per_class, int(proportional.get(c, 0))) for c in class_counts.index}
    target = {c: min(target[c], int(class_counts[c])) for c in target}

    total_target = sum(target.values())
    if total_target > max_samples:
        scale = max_samples / total_target
        target = {c: max(1, int(v * scale)) for c, v in target.items()}

    parts = []
    for c, n in target.items():
        pool = df[df["label"] == c]
        n = min(n, len(pool))
        parts.append(pool.sample(n=n, random_state=seed))

    out = pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

train_df = stratified_subsample(train_df, CONFIG["MAX_TRAIN_SAMPLES"], CONFIG["MIN_PER_CLASS"], SEED)
val_df = stratified_subsample(test_df, CONFIG["MAX_VAL_SAMPLES"], max(10, CONFIG["MIN_PER_CLASS"] // 4), SEED)

print(f"Train subset: {len(train_df)} / {len(train_df_full)} full")
print(f"Val monitoring subset: {len(val_df)} / {len(test_df_full)} full")

train_df["label"].map(GRADE_LABELS).value_counts().reindex(
    [GRADE_LABELS[i] for i in range(5)]).plot(kind="bar", title="Train SUBSET: DR grade distribution")
plt.ylabel("count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
def verify_images_exist(df, img_dir, n_check=20):
    missing = 0
    sample = df.sample(min(n_check, len(df)), random_state=SEED)
    for fn in sample["filename"]:
        p = Path(img_dir) / fn
        if not p.exists():
            alt = list(Path(img_dir).glob(Path(fn).stem + ".*"))
            if not alt:
                missing += 1
    print(f"{missing}/{len(sample)} sampled files missing in {img_dir}")

verify_images_exist(train_df, CONFIG["TRAIN_IMG_DIR"])
verify_images_exist(val_df, CONFIG["TEST_IMG_DIR"])


## 6. Dataset — quality-gated from ground truth, disk-cached, resumable, class-imbalance-aware

`GradingDataset` does four things the Stage 1 dataset didn't need to:
1. Builds a **single directory index** (stem → path) up front, instead of per-file
   `exists()`/`glob()` calls against Drive's FUSE mount.
2. **Drops every `Reject` row from `df` before any image is loaded** — using EyeQ's ground-truth
   `quality` column, not a Stage 1 model — so rejected images never cost you I/O time at all.
3. Applies CLAHE + illumination normalization to `Usable` images and passes `Good` through
   unchanged, resizing each image to `IMG_SIZE` right away.
4. **Writes each processed image straight to a fixed-record-size file on disk** (`DiskImageStore`)
   the moment it's decoded, instead of collecting the whole split as full-resolution arrays in a
   Python list first. That list-of-full-res-images step was the actual cause of the RAM crash —
   holding thousands of un-resized fundus images in memory at once, on top of the model and
   everything else, is what exhausted Colab's free-tier RAM. Now peak memory during a build is
   bounded by one in-flight batch (`num_load_workers` images), never by the size of the whole
   split — which is also what makes building the large, un-subsampled official test split in
   Section 11 safe on a CPU-only, free-tier runtime.

A small JSON manifest records which image indices have already been written, so if the runtime
disconnects or the process is killed mid-build, re-running only redoes the images that hadn't
been written yet — not the whole split from scratch.


In [ ]:
class DiskImageStore:
    """Fixed-record-size array of uint8 HxWxC images backed by a single file on disk, addressed
    by integer index. Uses plain seek/read/write (NOT numpy.memmap / mmap()) so it works
    reliably over Colab's Drive FUSE mount, which does not reliably support real memory-mapping.
    A single persistent file handle is reused for the object's whole lifetime, so there's no
    per-call open/close overhead."""

    def __init__(self, path, n, img_size, channels=3):
        self.path = Path(path)
        self.n = n
        self.img_size = img_size
        self.record_bytes = img_size * img_size * channels
        needed_size = self.record_bytes * n
        if not self.path.exists() or self.path.stat().st_size != needed_size:
            # (Re)create a correctly-sized file. This only throws away on-disk data when the
            # size doesn't match what this build expects (e.g. IMG_SIZE changed) -- see the
            # cache-tag naming below, which already keys the filename on (n, img_size, pipeline).
            self.path.parent.mkdir(parents=True, exist_ok=True)
            with open(self.path, "wb") as f:
                f.truncate(needed_size)
        self._fh = open(self.path, "r+b")

    def write(self, idx, img_rgb_uint8):
        self._fh.seek(idx * self.record_bytes)
        self._fh.write(np.ascontiguousarray(img_rgb_uint8, dtype=np.uint8).tobytes())

    def flush(self):
        self._fh.flush()
        try:
            os.fsync(self._fh.fileno())
        except OSError:
            pass  # some FUSE-backed mounts don't support fsync -- flush() above is enough

    def read(self, idx):
        self._fh.seek(idx * self.record_bytes)
        buf = self._fh.read(self.record_bytes)
        return np.frombuffer(buf, dtype=np.uint8).reshape(self.img_size, self.img_size, 3).copy()

    def close(self):
        try:
            self._fh.close()
        except Exception:
            pass


class _DiskImageView:
    """Read-through, list-like view over a DiskImageStore, remapped through `keep_idx` so
    `dataset.images_rgb[i]` lines up 1:1 with `dataset.labels[i]` even after unreadable images
    are dropped -- this keeps every downstream cell that indexes `.images_rgb` directly (Grad-CAM
    sanity checks, the report-building smoke test) working exactly as before."""

    def __init__(self, store, keep_idx):
        self._store = store
        self._keep_idx = keep_idx

    def __len__(self):
        return len(self._keep_idx)

    def __getitem__(self, i):
        return self._store.read(self._keep_idx[i])


class GradingDataset(Dataset):
    """Same public API as before (`.images_rgb`, `.labels`, `__getitem__`), but the processed
    images now live in a `DiskImageStore` instead of a Python list in RAM, and the build is
    resumable at the individual-image level. See Section 6's markdown above for why."""

    def __init__(self, df, img_dir, cache_dir, name, transform=None, apply_quality_pipeline=True,
                 num_index_workers=16, num_load_workers=8, batch_size=200):
        self.transform = transform
        self.img_dir = Path(img_dir)
        img_size = CONFIG["IMG_SIZE"]
        self._path_index = self._build_path_index(num_index_workers)

        if apply_quality_pipeline:
            n_before = len(df)
            df = df[df["quality"] != "Reject"].reset_index(drop=True)
            n_reject = n_before - len(df)
            print(f"Dropped {n_reject}/{n_before} ground-truth Reject images "
                  f"({n_reject/max(n_before,1)*100:.1f}%) before loading. {len(df)} remain.")

        rows = []
        for _, row in df.iterrows():
            p = self._resolve_path(row["filename"])
            if p is not None:
                rows.append((p, int(row["label"]), row.get("quality", "Good")))
        print(f"Resolved {len(rows)}/{len(df)} images in {self.img_dir.name}")

        n = len(rows)
        raw_labels = [y for _, y, _ in rows]

        cache_dir = Path(cache_dir)
        cache_dir.mkdir(parents=True, exist_ok=True)
        # cache tag bakes in everything that would invalidate a previous cache: split size,
        # resolution, and whether the quality gate/enhance step was applied.
        tag = f"{name}_{n}_{img_size}_{'qp' if apply_quality_pipeline else 'raw'}"
        store_path = cache_dir / f"{tag}.bin"
        manifest_path = cache_dir / f"{tag}_manifest.json"

        done, bad = set(), set()
        if manifest_path.exists():
            try:
                with open(manifest_path) as f:
                    manifest = json.load(f)
                done = set(manifest.get("done", []))
                bad = set(manifest.get("bad", []))
            except Exception as e:
                print(f"[{name}] Could not read manifest ({e}) -- rebuilding from scratch.")
                done, bad = set(), set()

        store = DiskImageStore(store_path, n, img_size)

        def _process_one(i):
            p, _, qlabel = rows[i]
            img_bgr = cv2.imread(str(p))
            if img_bgr is None:
                return i, None
            if apply_quality_pipeline:
                img_bgr = apply_quality_pipeline_fn(img_bgr, qlabel)
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            img_rgb = cv2.resize(img_rgb, (img_size, img_size), interpolation=cv2.INTER_AREA)
            return i, img_rgb

        remaining = [i for i in range(n) if i not in done and i not in bad]
        t0 = time.time()
        if remaining:
            skip_msg = f" ({len(done)} already cached from a previous run)" if done else ""
            print(f"[{name}] Processing {len(remaining)}/{n} images -> disk cache "
                  f"({store_path.name}){skip_msg}...")

        # Process in small batches: each batch is decoded/resized in parallel (I/O-bound, cv2
        # releases the GIL so threads help even on CPU), then written and checkpointed to the
        # manifest before the next batch starts. Peak RAM is ~num_load_workers full-resolution
        # images at a time, never the whole split.
        for chunk_start in range(0, len(remaining), batch_size):
            chunk = remaining[chunk_start: chunk_start + batch_size]
            with ThreadPoolExecutor(max_workers=num_load_workers) as ex:
                futs = [ex.submit(_process_one, i) for i in chunk]
                for fut in as_completed(futs):
                    i, img_rgb = fut.result()
                    if img_rgb is None:
                        bad.add(i)
                    else:
                        store.write(i, img_rgb)
                        done.add(i)
            store.flush()
            with open(manifest_path, "w") as f:
                json.dump({"done": sorted(done), "bad": sorted(bad)}, f)
            gc.collect()
            print(f"  ...{len(done)}/{n} done, {len(bad)} unreadable "
                  f"({time.time()-t0:.0f}s elapsed so far)")

        if remaining:
            print(f"[{name}] Finished: {len(done)}/{n} cached, {len(bad)} unreadable, "
                  f"took {time.time()-t0:.1f}s.")
        else:
            print(f"[{name}] All {len(done)}/{n} images already cached on disk -- nothing to do.")

        keep_idx = [i for i in range(n) if i in done]
        self.labels = [raw_labels[i] for i in keep_idx]
        self.images_rgb = _DiskImageView(store, keep_idx)
        self._store = store

    def _build_path_index(self, num_workers):
        t0 = time.time()
        index = {}
        for p in self.img_dir.iterdir():
            if p.is_file():
                index[p.stem] = p
        print(f"Indexed {len(index)} files in {self.img_dir.name} in {time.time()-t0:.1f}s")
        return index

    def _resolve_path(self, filename):
        stem = Path(filename).stem
        return self._path_index.get(stem)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images_rgb[idx]
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

# alias so the class body above can call the module-level enhancement function
# without shadowing it via a same-named parameter
apply_quality_pipeline_fn = apply_quality_pipeline


### Persistent dataset cache (survives Colab disconnects AND runs on CPU-safe RAM)

Building `train_dataset`/`val_dataset`/`test_dataset_full` is the slow part (image I/O), and a
free-tier session can drop -- or run out of RAM -- at any time. `GradingDataset` now writes each
processed image straight to a `DiskImageStore` file on Drive as soon as it's decoded (Section 6),
checkpointing a small JSON manifest after every batch. `get_or_build_dataset()` below is now just
a thin wrapper that points `GradingDataset` at the shared `CACHE_DIR` -- the resumability lives in
the dataset class itself, at the individual-image level, so:

- Re-running after a disconnect or a crash only reprocesses the images that weren't finished yet,
  never the whole split.
- RAM use during a build no longer scales with the size of the split (previously, holding every
  full-resolution decoded image in a Python list before resizing was what exhausted Colab's
  free-tier RAM) -- it's bounded by one in-flight batch instead.


In [ ]:
CACHE_DIR = Path(CONFIG["OUTPUT_DIR"]) / "dataset_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def get_or_build_dataset(name, df, img_dir, transform, apply_quality_pipeline):
    """Thin wrapper kept for call-site compatibility with the rest of the notebook --
    `GradingDataset` itself now owns the on-disk, resumable, RAM-safe caching (see Section 6's
    docstring/markdown), so this just wires the shared `CACHE_DIR` in."""
    return GradingDataset(df, img_dir, cache_dir=CACHE_DIR, name=name, transform=transform,
                           apply_quality_pipeline=apply_quality_pipeline)


In [ ]:
train_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"])),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),           # fundus images are rotation-tolerant, but keep it mild
    T.ColorJitter(brightness=0.1, contrast=0.1),   # mild — don't wash out small lesions
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"])),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_dataset = get_or_build_dataset("train", train_df, CONFIG["TRAIN_IMG_DIR"], train_transform,
                                      CONFIG["APPLY_QUALITY_PIPELINE"])
val_dataset = get_or_build_dataset("val", val_df, CONFIG["TEST_IMG_DIR"], eval_transform,
                                    CONFIG["APPLY_QUALITY_PIPELINE"])


## 7. Weighted sampling + effective-number class weights

Two complementary imbalance handles, matching the Stage 1 pattern but with a stronger
weighting scheme since EyeQ's grade skew is worse than its quality skew (grade 0 vs grade 4
can be a >20:1 ratio, vs. quality's milder split):

- **`WeightedRandomSampler`** — balances what the model *sees* per batch.
- **Effective-Number class weights in the loss** — a second guard so even within a batch the
  loss doesn't get dominated by majority-class errors. Effective number
  `E_n = (1-beta^n)/(1-beta)` (Cui et al., 2019) is smoother than raw inverse-frequency, which
  tends to overweight the rarest class so much it destabilizes training on a small subset.


In [ ]:
labels_arr = np.array(train_dataset.labels)
class_counts = pd.Series(labels_arr).value_counts().sort_index()
# make sure all 5 classes are represented in the index even if a class got zero samples
class_counts = class_counts.reindex(range(CONFIG["NUM_CLASSES"]), fill_value=0)
print("Effective train-subset class counts (post Stage-1 filtering):")
print(class_counts.rename(index=GRADE_LABELS))

sample_weight_map = (1.0 / class_counts.replace(0, 1)).to_dict()
sample_weights = [sample_weight_map[y] for y in labels_arr]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

beta = CONFIG["CB_BETA"]
effective_num = 1.0 - np.power(beta, class_counts.values.astype(np.float64))
effective_num = np.where(effective_num == 0, 1e-6, effective_num)  # guard zero-count classes
cb_weights = (1.0 - beta) / effective_num
cb_weights = cb_weights / cb_weights.sum() * CONFIG["NUM_CLASSES"]
loss_weights = torch.tensor(cb_weights, dtype=torch.float32).to(device)
print("Class-balanced loss weights:", dict(zip(GRADE_LABELS.values(), cb_weights.round(3))))

_loader_kwargs = dict(num_workers=CONFIG["NUM_WORKERS"], pin_memory=PIN_MEMORY,
                       persistent_workers=CONFIG["NUM_WORKERS"] > 0)
if CONFIG["NUM_WORKERS"] > 0:
    _loader_kwargs["prefetch_factor"] = CONFIG["PREFETCH_FACTOR"]

train_loader = DataLoader(train_dataset, batch_size=CONFIG["BATCH_SIZE"], sampler=sampler, **_loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, **_loader_kwargs)


## 8. Model — EfficientNet-B3 backbone, CORAL ordinal head (or plain softmax fallback)

**CORAL** (COnsistent RAnk Logits, Cao et al. 2020) turns 5-class ordinal grading into
`num_classes-1` binary "is grade > k?" tasks that share the backbone and share weights except
for per-threshold biases. This structurally forbids nonsensical predictions like "grade 3 more
likely than grade 2 AND grade 1" and empirically reduces the adjacent-grade confusions the
architecture doc calls out (0↔1, 2↔3) compared to plain softmax, without needing extra
parameters worth mentioning.


In [ ]:
class CoralHead(nn.Module):
    """Shared single-logit trunk + per-threshold bias, per Cao et al. 2020."""
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.num_thresholds = num_classes - 1
        self.fc = nn.Linear(in_features, 1, bias=False)
        self.biases = nn.Parameter(torch.zeros(self.num_thresholds))

    def forward(self, x):
        logit = self.fc(x)                      # (B, 1)
        return logit + self.biases               # (B, num_thresholds) -- broadcasts


def coral_probs(threshold_logits):
    """Robust CORAL class-probability conversion (independent-sigmoid approximation, the
    standard practical decoding used in the CORAL/CORN literature): P(grade=k) is built from
    consecutive differences of P(grade>k-1) with clamping + renormalization so it's always a
    valid distribution even though the underlying sigmoids aren't perfectly consistent early
    in training."""
    p_greater = torch.sigmoid(threshold_logits)              # (B, K-1), P(grade > k) for k=0..K-2
    B = p_greater.size(0)
    ones = torch.ones(B, 1, device=p_greater.device)
    zeros = torch.zeros(B, 1, device=p_greater.device)
    p_greater_ext = torch.cat([ones, p_greater, zeros], dim=1)   # P(grade > -1)=1 ... P(grade > K-1)=0
    class_probs = p_greater_ext[:, :-1] - p_greater_ext[:, 1:]     # (B, K)
    class_probs = class_probs.clamp(min=1e-6)
    class_probs = class_probs / class_probs.sum(dim=1, keepdim=True)
    return class_probs


def coral_predicted_grade(threshold_logits):
    """Rank-consistent prediction: count how many thresholds are exceeded (P>0.5 each)."""
    return (torch.sigmoid(threshold_logits) > 0.5).sum(dim=1)


class CoralLoss(nn.Module):
    """Sum of per-threshold weighted BCE, per Cao et al. Class weighting is applied at the
    sample level (weight of the sample's true class) since CORAL's per-threshold labels don't
    map 1:1 onto the 5 original classes."""
    def __init__(self, num_thresholds, class_weights=None):
        super().__init__()
        self.num_thresholds = num_thresholds
        self.class_weights = class_weights  # tensor (num_classes,) or None

    def forward(self, threshold_logits, targets):
        # binary target for each threshold k: 1 if true grade > k else 0
        levels = torch.arange(self.num_thresholds, device=targets.device).unsqueeze(0)
        binary_targets = (targets.unsqueeze(1) > levels).float()          # (B, K-1)
        loss_per_threshold = F.binary_cross_entropy_with_logits(
            threshold_logits, binary_targets, reduction="none").sum(dim=1)  # (B,)
        if self.class_weights is not None:
            sample_w = self.class_weights[targets]
            loss_per_threshold = loss_per_threshold * sample_w
        return loss_per_threshold.mean()


class GradingModel(nn.Module):
    """timm backbone (features only) + either a CORAL head or a plain linear softmax head."""
    def __init__(self, arch, num_classes, ordinal=True, pretrained=True):
        super().__init__()
        self.ordinal = ordinal
        self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0)  # pooled features
        feat_dim = self.backbone.num_features
        if ordinal:
            self.head = CoralHead(feat_dim, num_classes)
        else:
            self.head = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats)

    def forward_features_for_cam(self, x):
        """Returns the last conv feature map (pre-pool) for Grad-CAM, plus pooled features."""
        feats_map = self.backbone.forward_features(x)   # (B, C, H, W)
        pooled = self.backbone.forward_head(feats_map, pre_logits=True)
        return feats_map, pooled


def build_model():
    m = GradingModel(CONFIG["MODEL_ARCH"], CONFIG["NUM_CLASSES"],
                      ordinal=CONFIG["USE_ORDINAL_HEAD"], pretrained=True)
    return m.to(device)


def set_backbone_trainable(model, trainable):
    for p in model.backbone.parameters():
        p.requires_grad = trainable
    for p in model.head.parameters():
        p.requires_grad = True  # head always trainable


model = build_model()
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{CONFIG['MODEL_ARCH']} (ordinal={CONFIG['USE_ORDINAL_HEAD']}) — {n_params/1e6:.2f}M trainable params (full)")

if CONFIG["FREEZE_BACKBONE_EPOCHS"] > 0:
    set_backbone_trainable(model, trainable=False)
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Backbone frozen for the first {CONFIG['FREEZE_BACKBONE_EPOCHS']} epochs "
          f"({n_trainable/1e6:.2f}M trainable params, head only).")


## 9. Loss, optimizer, epoch loop (with quadratic-weighted kappa tracked alongside accuracy)

In [ ]:
if CONFIG["USE_ORDINAL_HEAD"]:
    criterion = CoralLoss(num_thresholds=CONFIG["NUM_CLASSES"] - 1, class_weights=loss_weights)
else:
    criterion = nn.CrossEntropyLoss(weight=loss_weights, label_smoothing=CONFIG["LABEL_SMOOTHING"])

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["EPOCHS"])
scaler = GradScaler(enabled=CONFIG["USE_AMP"])


def predict_grade_and_probs(outputs):
    if CONFIG["USE_ORDINAL_HEAD"]:
        probs = coral_probs(outputs)
        preds = coral_predicted_grade(outputs)
    else:
        probs = F.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)
    return preds, probs


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    torch.set_grad_enabled(train)
    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=CONFIG["USE_AMP"]):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        if train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * imgs.size(0)
        preds, _ = predict_grade_and_probs(outputs)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    qwk = cohen_kappa_score(all_labels, all_preds, weights="quadratic")
    return avg_loss, acc, qwk


### Quick timing dry-run (right-size `MAX_TRAIN_SAMPLES` / `EPOCHS` / `IMG_SIZE` before committing)

In [ ]:
n_probe_batches = min(10, len(train_loader))
t0 = time.time()
it = iter(train_loader)
for _ in range(n_probe_batches):
    imgs, labels = next(it)
    imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
    optimizer.zero_grad(set_to_none=True)
    with autocast(enabled=CONFIG["USE_AMP"]):
        outputs = model(imgs)
        loss = criterion(outputs, labels)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
dt = time.time() - t0

sec_per_batch = dt / n_probe_batches
batches_per_epoch = len(train_loader) + len(val_loader)
est_epoch_s = sec_per_batch * batches_per_epoch
est_total_min = est_epoch_s * CONFIG["EPOCHS"] / 60

print(f"~{sec_per_batch:.2f}s/batch -> ~{est_epoch_s:.0f}s/epoch -> "
      f"~{est_total_min:.1f} min for all {CONFIG['EPOCHS']} epochs (early stopping usually cuts this shorter)")
budget_min = CONFIG["TIME_BUDGET_MINUTES"]
verdict = "looks comfortable" if est_total_min < budget_min else "consider lowering MAX_TRAIN_SAMPLES, EPOCHS, BATCH_SIZE, or IMG_SIZE"
print(f"Configured soft time budget: {budget_min} min — {verdict}")


## 10. Train

Same progressive-unfreeze / resume-from-checkpoint / early-stopping pattern as Stage 1, tracking
**quadratic-weighted kappa** (the standard DR-grading metric) alongside accuracy, and selecting
the best checkpoint by whichever `CONFIG["EARLY_STOP_METRIC"]` you chose.


**Persistence note:** this notebook now runs on CPU by design (`FORCE_CPU = True` in Section 2) so a missing/expired free-tier GPU quota can never be the reason a run dies -- it just takes longer, on purpose. On top of that, two checkpoints get written to Drive every epoch, not just on
improvement — `last.pt` (latest state, always overwritten) and `model.pt` (best-score state
only). If the free-tier session disconnects mid-training, just re-run this notebook top to
bottom: the dataset cache (Section 6) skips re-processing, and this cell **auto-resumes from
`last.pt`** with no prompt needed, so nothing is lost except the epoch(s) that were mid-flight
when it died.

In [ ]:
history = {"train_loss": [], "train_acc": [], "train_qwk": [],
           "val_loss": [], "val_acc": [], "val_qwk": []}
best_score = -1.0
patience_counter = 0
start_epoch = 1
ckpt_path = Path(CONFIG["OUTPUT_DIR"]) / "model.pt"          # best-score checkpoint (used for eval/export)
last_ckpt_path = Path(CONFIG["OUTPUT_DIR"]) / "last.pt"        # most-recent checkpoint (used to resume)
history_path = Path(CONFIG["OUTPUT_DIR"]) / "training_history.json"

def _load_checkpoint(path):
    try:
        return torch.load(path, map_location=device)
    except Exception as e:
        print(f"Could not load checkpoint at {path} ({type(e).__name__}: {e}) — ignoring it.")
        return None

resume_ckpt = _load_checkpoint(last_ckpt_path) if last_ckpt_path.exists() else None
if resume_ckpt is None and ckpt_path.exists():
    resume_ckpt = _load_checkpoint(ckpt_path)   # fall back to best.pt if last.pt is missing/corrupt

if resume_ckpt is not None:
    model.load_state_dict(resume_ckpt["model_state_dict"])
    if "optimizer_state_dict" in resume_ckpt:
        try:
            optimizer.load_state_dict(resume_ckpt["optimizer_state_dict"])
            scheduler.load_state_dict(resume_ckpt["scheduler_state_dict"])
        except Exception as e:
            print(f"Optimizer/scheduler state didn't restore cleanly ({e}) — continuing with fresh ones.")
    best_score = resume_ckpt.get("best_score", resume_ckpt.get("score", -1.0))
    start_epoch = resume_ckpt.get("epoch", 0) + 1
    if history_path.exists():
        with open(history_path) as f:
            history = json.load(f)
    print(f"Auto-resumed from epoch {resume_ckpt.get('epoch')} (best_score={best_score:.4f}). "
          f"Continuing from epoch {start_epoch}.")
    if start_epoch > CONFIG["FREEZE_BACKBONE_EPOCHS"]:
        set_backbone_trainable(model, trainable=True)
else:
    print("No existing checkpoint found — starting training from scratch.")

session_start = time.time()
time_budget_s = CONFIG["TIME_BUDGET_MINUTES"] * 60
metric_key = "qwk" if CONFIG["EARLY_STOP_METRIC"] == "qwk" else "acc"

if start_epoch > CONFIG["EPOCHS"]:
    print(f"start_epoch ({start_epoch}) already exceeds CONFIG['EPOCHS'] ({CONFIG['EPOCHS']}) — "
          f"training is already complete for this config. Raise EPOCHS in CONFIG to keep going, "
          f"or skip ahead to Section 11 (evaluation).")

for epoch in range(start_epoch, CONFIG["EPOCHS"] + 1):
    if CONFIG["FREEZE_BACKBONE_EPOCHS"] > 0 and epoch == CONFIG["FREEZE_BACKBONE_EPOCHS"] + 1:
        set_backbone_trainable(model, trainable=True)
        print(f"[epoch {epoch}] Unfroze backbone — fine-tuning the whole network now.")

    train_loss, train_acc, train_qwk = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_qwk = run_epoch(val_loader, train=False)
    scheduler.step()

    history["train_loss"].append(train_loss); history["train_acc"].append(train_acc); history["train_qwk"].append(train_qwk)
    history["val_loss"].append(val_loss); history["val_acc"].append(val_acc); history["val_qwk"].append(val_qwk)

    score = val_qwk if metric_key == "qwk" else val_acc
    improved = score > best_score
    print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} acc {train_acc:.3f} qwk {train_qwk:.3f} "
          f"| val loss {val_loss:.4f} acc {val_acc:.3f} qwk {val_qwk:.3f} {'*' if improved else ''}")

    if improved:
        best_score = score
        patience_counter = 0
    else:
        patience_counter += 1

    # ALWAYS persist the latest state (not just on improvement) so a mid-session disconnect
    # loses at most the epoch in progress, never the whole run.
    checkpoint_payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "epoch": epoch, "score": score, "best_score": best_score,
        "val_acc": val_acc, "val_qwk": val_qwk,
    }
    torch.save(checkpoint_payload, last_ckpt_path)
    if improved:
        torch.save(checkpoint_payload, ckpt_path)

    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)

    if patience_counter >= CONFIG["EARLY_STOP_PATIENCE"]:
        print(f"Early stopping at epoch {epoch} (no improvement in {CONFIG['EARLY_STOP_PATIENCE']} epochs).")
        break

    if time.time() - session_start > time_budget_s:
        print(f"Hit the {CONFIG['TIME_BUDGET_MINUTES']}-min soft time budget — stopping here. "
              f"Re-run this cell later to resume from the saved checkpoint.")
        break

print(f"Best val {metric_key}: {best_score:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history["train_loss"], label="train"); axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(history["train_acc"], label="train"); axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()

axes[2].plot(history["train_qwk"], label="train"); axes[2].plot(history["val_qwk"], label="val")
axes[2].axhline(CONFIG["BASELINE_QWK"], color="gray", linestyle="--", label="ballpark baseline")
axes[2].set_title("Quadratic-Weighted Kappa"); axes[2].set_xlabel("epoch"); axes[2].legend()
plt.tight_layout()
plt.show()


## 11. Calibrate the referable-DR operating point, then evaluate on the FULL official test split

Same discipline as Stage 1: training/early-stopping used a small subset purely for speed;
the number you actually report comes from a single inference pass over **every image** in
EyeQ's official test split. Two extra pieces vs. Stage 1, both from the architecture doc:

- **Temperature scaling** for calibrated confidence (fit on the val subset, applied at test time).
- **Referable-DR threshold search** on `P(grade >= REFERABLE_GRADE_CUTOFF)` — tuned for
  ≥90% sensitivity while keeping ≥85% specificity, **not** the naive argmax operating point.


In [ ]:
def referable_prob_from_class_probs(class_probs, cutoff):
    """class_probs: (N, 5) softmax-like distribution over grades 0-4."""
    return class_probs[:, cutoff:].sum(axis=1)


class TemperatureScaler(nn.Module):
    """Fits a single scalar temperature on held-out logits to calibrate confidence. Only
    meaningful for the plain-softmax head; for the CORAL head we scale the threshold logits
    the same way before converting to class probabilities."""
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature.clamp(min=0.05)


@torch.no_grad()
def collect_logits(loader):
    model.eval()
    all_logits, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        with autocast(enabled=CONFIG["USE_AMP"]):
            outputs = model(imgs)
        all_logits.append(outputs.float().cpu())
        all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)


val_logits, val_labels = collect_logits(val_loader)

temp_scaler = TemperatureScaler()
opt_t = torch.optim.LBFGS([temp_scaler.temperature], lr=0.05, max_iter=100)
nll = nn.CrossEntropyLoss() if not CONFIG["USE_ORDINAL_HEAD"] else None

if CONFIG["USE_ORDINAL_HEAD"]:
    # simple grid search for CORAL temperature (LBFGS-on-BCE also works; grid is simpler & robust)
    best_T, best_nll = 1.0, float("inf")
    for T_cand in np.linspace(0.5, 3.0, 26):
        probs = coral_probs(val_logits / T_cand)
        nll_val = F.nll_loss(torch.log(probs.clamp(min=1e-8)), val_labels).item()
        if nll_val < best_nll:
            best_nll, best_T = nll_val, T_cand
    temp_scaler.temperature.data = torch.tensor([best_T])
    print(f"Fitted CORAL temperature: {best_T:.3f} (val NLL {best_nll:.4f})")
else:
    def _closure():
        opt_t.zero_grad()
        loss = nll(temp_scaler(val_logits), val_labels)
        loss.backward()
        return loss
    opt_t.step(_closure)
    print(f"Fitted softmax temperature: {temp_scaler.temperature.item():.3f}")


def logits_to_calibrated_class_probs(logits):
    scaled = logits / temp_scaler.temperature.clamp(min=0.05)
    if CONFIG["USE_ORDINAL_HEAD"]:
        return coral_probs(scaled).numpy()
    return F.softmax(scaled, dim=1).numpy()


val_class_probs = logits_to_calibrated_class_probs(val_logits)
val_ref_prob = referable_prob_from_class_probs(val_class_probs, CONFIG["REFERABLE_GRADE_CUTOFF"])
val_ref_true = (val_labels.numpy() >= CONFIG["REFERABLE_GRADE_CUTOFF"]).astype(int)

fpr, tpr, thresh = roc_curve(val_ref_true, val_ref_prob)
sens = tpr
spec = 1 - fpr

candidates = [(t, s, sp) for t, s, sp in zip(thresh, sens, spec)
              if s >= CONFIG["TARGET_SENSITIVITY"] and sp >= CONFIG["MIN_SPECIFICITY"]]
if candidates:
    # among valid candidates, prefer the highest specificity (still hitting target sensitivity)
    best_threshold, best_sens, best_spec = max(candidates, key=lambda c: c[2])
else:
    # fall back: hit the sensitivity target as closely as possible, report the tradeoff honestly
    idx = np.argmin(np.abs(sens - CONFIG["TARGET_SENSITIVITY"]))
    best_threshold, best_sens, best_spec = thresh[idx], sens[idx], spec[idx]
    print("WARNING: could not hit both the sensitivity and specificity targets simultaneously "
          "on the validation subset — falling back to the closest sensitivity match. "
          "This usually means more training data / epochs / a stronger backbone is needed; "
          "re-check once you can afford a larger MAX_TRAIN_SAMPLES.")

print(f"Calibrated referable-DR threshold: P(grade>={CONFIG['REFERABLE_GRADE_CUTOFF']}) >= {best_threshold:.3f}")
print(f"  -> val sensitivity {best_sens:.3f}, val specificity {best_spec:.3f} "
      f"(targets: sens>={CONFIG['TARGET_SENSITIVITY']}, spec>={CONFIG['MIN_SPECIFICITY']})")


In [ ]:
# ---- Full official test-set evaluation (the number that's actually reported) ----
test_dataset_full = get_or_build_dataset("test_full", test_df_full, CONFIG["TEST_IMG_DIR"], eval_transform,
                                          CONFIG["APPLY_QUALITY_PIPELINE"])
test_loader_full = DataLoader(test_dataset_full, batch_size=CONFIG["BATCH_SIZE"], shuffle=False,
                               num_workers=CONFIG["NUM_WORKERS"], pin_memory=PIN_MEMORY)

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

test_logits, test_labels = collect_logits(test_loader_full)
test_class_probs = logits_to_calibrated_class_probs(test_logits)
test_preds = test_class_probs.argmax(axis=1)
test_labels_np = test_labels.numpy()

test_acc = accuracy_score(test_labels_np, test_preds)
test_qwk = cohen_kappa_score(test_labels_np, test_preds, weights="quadratic")
report = classification_report(test_labels_np, test_preds, target_names=[GRADE_LABELS[i] for i in range(5)],
                                output_dict=True, zero_division=0)
cm = confusion_matrix(test_labels_np, test_preds)

test_ref_true = (test_labels_np >= CONFIG["REFERABLE_GRADE_CUTOFF"]).astype(int)
test_ref_prob = referable_prob_from_class_probs(test_class_probs, CONFIG["REFERABLE_GRADE_CUTOFF"])
test_ref_pred = (test_ref_prob >= best_threshold).astype(int)

tp = int(((test_ref_pred == 1) & (test_ref_true == 1)).sum())
tn = int(((test_ref_pred == 0) & (test_ref_true == 0)).sum())
fp = int(((test_ref_pred == 1) & (test_ref_true == 0)).sum())
fn = int(((test_ref_pred == 0) & (test_ref_true == 1)).sum())
test_sensitivity = tp / (tp + fn) if (tp + fn) else float("nan")
test_specificity = tn / (tn + fp) if (tn + fp) else float("nan")
test_auc = roc_auc_score(test_ref_true, test_ref_prob) if len(set(test_ref_true)) > 1 else float("nan")

print(f"Full official test-set accuracy: {test_acc:.4f}")
print(f"Full official test-set QWK:      {test_qwk:.4f}  vs ballpark baseline {CONFIG['BASELINE_QWK']}")
print(f"Referable DR (grade>={CONFIG['REFERABLE_GRADE_CUTOFF']}) at calibrated threshold {best_threshold:.3f}:")
print(f"  sensitivity = {test_sensitivity:.4f}  (target >= {CONFIG['TARGET_SENSITIVITY']})")
print(f"  specificity = {test_specificity:.4f}  (target >= {CONFIG['MIN_SPECIFICITY']})")
print(f"  ROC-AUC     = {test_auc:.4f}")
print(f"(Trained on a {len(train_dataset)}-image quality-filtered/enhanced subset of the "
      f"{len(train_df_full)}-image full EyeQ train set — evaluated on all {len(test_dataset_full)} "
      f"quality-filtered official test images.)")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
target_names = [GRADE_LABELS[i] for i in range(5)]
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(5)); ax.set_xticklabels(target_names, rotation=30, ha="right")
ax.set_yticks(range(5)); ax.set_yticklabels(target_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(5):
    for j in range(5):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.colorbar(im)
plt.title(f"5-class confusion matrix — full test set (QWK={test_qwk:.3f})")
plt.tight_layout()
plt.show()


In [ ]:
eval_report = {
    "test_accuracy": float(test_acc),
    "test_qwk": float(test_qwk),
    "baseline_qwk": CONFIG["BASELINE_QWK"],
    "beats_baseline_qwk": bool(test_qwk >= CONFIG["BASELINE_QWK"]),
    "referable_grade_cutoff": CONFIG["REFERABLE_GRADE_CUTOFF"],
    "referable_threshold_prob": float(best_threshold),
    "referable_sensitivity": float(test_sensitivity),
    "referable_specificity": float(test_specificity),
    "referable_roc_auc": float(test_auc),
    "meets_sensitivity_target": bool(test_sensitivity >= CONFIG["TARGET_SENSITIVITY"]),
    "meets_specificity_target": bool(test_specificity >= CONFIG["MIN_SPECIFICITY"]),
    "classification_report": report,
    "confusion_matrix": cm.tolist(),
    "class_order": target_names,
    "train_subset_size": len(train_dataset),
    "train_full_size": len(train_df_full),
    "val_monitoring_subset_size": len(val_dataset),
    "test_set_size_full": len(test_dataset_full),
    "used_subset_for_training": bool(CONFIG["USE_SUBSET"]),
    "used_ordinal_head": bool(CONFIG["USE_ORDINAL_HEAD"]),
    "quality_pipeline_applied": bool(CONFIG["APPLY_QUALITY_PIPELINE"]),
}
with open(Path(CONFIG["OUTPUT_DIR"]) / "eval_report.json", "w") as f:
    json.dump(eval_report, f, indent=2)
print("Saved eval_report.json")
print(json.dumps(eval_report, indent=2)[:1500], "...")


## 12. Grad-CAM — explainability output for the report

Grad-CAM is computed against the **referable-DR score** `P(grade >= REFERABLE_GRADE_CUTOFF)`
rather than the raw argmax class logit — that's the quantity a clinician actually needs
justified ("why did this get flagged as referable?"), and it's stable regardless of whether
you're using the ordinal or softmax head.


In [ ]:
class GradCAM:
    def __init__(self, model):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer = model.backbone.conv_head if hasattr(model.backbone, "conv_head") \
            else list(model.backbone.children())[-1]
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, x):
        """x: (1, 3, H, W) preprocessed tensor. Returns a (H, W) heatmap in [0, 1]."""
        self.model.eval()
        x = x.clone().requires_grad_(True)
        feats_map, pooled = self.model.forward_features_for_cam(x)
        # re-run head manually on the *hooked* activations path so gradients flow correctly
        outputs = self.model.head(pooled)
        if CONFIG["USE_ORDINAL_HEAD"]:
            probs = coral_probs(outputs / temp_scaler.temperature.clamp(min=0.05))
        else:
            probs = F.softmax(outputs / temp_scaler.temperature.clamp(min=0.05), dim=1)
        score = probs[:, CONFIG["REFERABLE_GRADE_CUTOFF"]:].sum()

        self.model.zero_grad()
        score.backward()

        grads = self.gradients[0]                      # (C, h, w)
        acts = self.activations[0]                      # (C, h, w)
        weights = grads.mean(dim=(1, 2))                 # (C,) global-average-pooled gradients
        cam = torch.relu((weights[:, None, None] * acts).sum(dim=0))
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy(), float(score.item())


gradcam = GradCAM(model)


def overlay_gradcam(image_rgb_uint8, cam, alpha=0.4):
    h, w = image_rgb_uint8.shape[:2]
    cam_resized = cv2.resize(cam, (w, h))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    return np.uint8(image_rgb_uint8 * (1 - alpha) + heatmap * alpha)


In [ ]:
# Sanity-check on a few validation images, saved for visual review
n_show = min(4, len(val_dataset))
fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
for i in range(n_show):
    img_rgb = val_dataset.images_rgb[i]
    x = eval_transform(img_rgb).unsqueeze(0).to(device)
    cam, ref_score = gradcam(x)
    img_resized = cv2.resize(img_rgb, (CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"]))
    overlay = overlay_gradcam(img_resized, cam)

    true_grade = GRADE_LABELS[val_dataset.labels[i]]
    axes[0, i].imshow(img_resized); axes[0, i].set_title(f"True: {true_grade}"); axes[0, i].axis("off")
    axes[1, i].imshow(overlay); axes[1, i].set_title(f"Grad-CAM (ref score={ref_score:.2f})"); axes[1, i].axis("off")

    Image.fromarray(overlay).save(Path(CONFIG["OUTPUT_DIR"]) / "gradcam_samples" / f"sample_{i}.png")

plt.tight_layout()
plt.show()
print("Saved sample overlays to", Path(CONFIG["OUTPUT_DIR"]) / "gradcam_samples")


## 13. Export for pipeline integration

In [ ]:
export_config = {
    "stage": "stage3_grading",
    "model_arch": CONFIG["MODEL_ARCH"],
    "ordinal_head": CONFIG["USE_ORDINAL_HEAD"],
    "num_classes": CONFIG["NUM_CLASSES"],
    "img_size": CONFIG["IMG_SIZE"],
    "normalize_mean": IMAGENET_MEAN,
    "normalize_std": IMAGENET_STD,
    "class_idx_to_label": GRADE_LABELS,
    "referable_grade_cutoff": CONFIG["REFERABLE_GRADE_CUTOFF"],
    "referable_threshold_prob": float(best_threshold),
    "temperature": float(temp_scaler.temperature.item()),
    "input_pipeline": "expects a Stage-1-processed image: Reject already dropped upstream, "
                       "Usable already CLAHE+illumination-normalized, Good passed through raw",
    "test_accuracy": float(test_acc),
    "test_qwk": float(test_qwk),
    "test_referable_sensitivity": float(test_sensitivity),
    "test_referable_specificity": float(test_specificity),
    "checkpoint_file": "model.pt",
    "trained_on": f"EyeQ official train split — Stage-1-filtered/enhanced stratified "
                  f"{len(train_dataset)}-image subset (full split has {len(train_df_full)} images)",
    "evaluated_on": f"EyeQ official FULL test split, Stage-1-processed ({len(test_dataset_full)} images)",
}
with open(Path(CONFIG["OUTPUT_DIR"]) / "config.json", "w") as f:
    json.dump(export_config, f, indent=2, default=str)

print("Exported to:", CONFIG["OUTPUT_DIR"])
print(json.dumps(export_config, indent=2, default=str))


## 14. Report assembly — merging Stage 1 + Stage 3/4 + (separately-run) Stage 2

Per the updated architecture, Stage 2's masks are **not** a model input anymore — they get
folded into the final report alongside this model's grade + Grad-CAM. This helper shows the
merge shape the FastAPI `/reports/{report_id}` endpoint should assemble; it works with or
without a Stage 2 mask bundle, since Stage 2 runs as its own (separate) service call now.


In [ ]:
def build_report(image_rgb, stage1_quality_result, stage2_mask_bundle=None):
    """stage1_quality_result: {'label','confidence','probs'} from the Stage 1 notebook's
    predict_quality(). stage2_mask_bundle: optional dict from the Stage 2 notebook
    ({'lesion_masks','od_mask','macula_point','vessel_mask'}) — purely for display/evidence
    in the report, not consumed by this model."""
    x = eval_transform(image_rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        with autocast(enabled=CONFIG["USE_AMP"]):
            outputs = model(x)
        scaled = outputs / temp_scaler.temperature.clamp(min=0.05)
        class_probs = (coral_probs(scaled) if CONFIG["USE_ORDINAL_HEAD"]
                       else F.softmax(scaled, dim=1)).cpu().numpy()[0]
    grade = int(class_probs.argmax())
    ref_prob = float(class_probs[CONFIG["REFERABLE_GRADE_CUTOFF"]:].sum())
    referable = ref_prob >= best_threshold

    cam, _ = gradcam(x)
    img_resized = cv2.resize(image_rgb, (CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"]))
    gradcam_overlay = overlay_gradcam(img_resized, cam)

    report = {
        "stage1_quality": stage1_quality_result,
        "stage3_grade": {
            "icdr_grade": grade,
            "icdr_label": GRADE_LABELS[grade],
            "class_probabilities": {GRADE_LABELS[i]: float(p) for i, p in enumerate(class_probs)},
            "referable": bool(referable),
            "referable_probability": ref_prob,
            "referable_threshold": float(best_threshold),
        },
        "stage4_explainability": {
            "gradcam_overlay": gradcam_overlay,   # HxWx3 uint8 array — save/encode as needed
            "note": "Grad-CAM computed w.r.t. the referable-DR score, not raw argmax class.",
        },
        "stage2_segmentation": stage2_mask_bundle if stage2_mask_bundle is not None else
            {"note": "Stage 2 not provided to this call — run it separately and merge before rendering."},
    }
    return report


# Smoke test on one validation image
_sample_report = build_report(
    val_dataset.images_rgb[0],
    stage1_quality_result={"label": "Good", "confidence": 0.95, "probs": {"Good": 0.95, "Usable": 0.04, "Reject": 0.01}},
)
print("Grade:", _sample_report["stage3_grade"]["icdr_label"],
      "| Referable:", _sample_report["stage3_grade"]["referable"],
      "| P(referable):", round(_sample_report["stage3_grade"]["referable_probability"], 3))
plt.imshow(_sample_report["stage4_explainability"]["gradcam_overlay"])
plt.title("Stage 4 Grad-CAM overlay (sample)")
plt.axis("off")
plt.show()


## Notes for integrating into the FastAPI backend

- Copy the whole `stage3_grading/` folder (from `OUTPUT_DIR`) into the backend's model store,
  next to `stage1_quality/`.
- The Stage 3 service should load `config.json` + `model.pt`, then apply the exact
  `input_pipeline` described in `config.json` (Stage 1 gate/enhance first) before calling this
  model — that contract is what `POST /images/{id}/analyze` should already be doing per the
  architecture doc.
- `GET /reports/{report_id}` should assemble the response the same shape `build_report()`
  returns above: Stage 1 quality result + Stage 3 grade/referable flag/confidence + Stage 4
  Grad-CAM overlay, merged with a separately-computed Stage 2 mask bundle (looked up by image
  id, not recomputed here) — exactly the "combine at report time, not at model-input time"
  split this notebook implements.
- If sensitivity/specificity targets aren't met on the full test set: the first lever to pull
  is `MAX_TRAIN_SAMPLES` (more data), the second is `EPOCHS`/`FREEZE_BACKBONE_EPOCHS` (more
  fine-tuning), and only then consider a larger backbone (`efficientnet_b4`) — in that order,
  since on Colab free tier, data and training budget are almost always the binding constraint
  before model capacity is.
- `CONFIG["REFERABLE_GRADE_CUTOFF"]` is a single knob — set it to `1` if you need to match the
  earlier architecture-doc draft instead of the Functionalities-doc definition (Level 2+); keep
  whichever one your clinical validation partner expects, and say explicitly in the eval report
  which convention was used (this notebook's `eval_report.json` already records it).
